# Ensamble de corridas independientes del clasificador K=70

Combina modelos **entrenados por separado** (cada uno con su notebook) promediando sus probabilidades,
$\bar p = \frac{1}{M}\sum_m p_m$, y mide cuánto mejora cada combinación. A diferencia del snapshot ensemble
(`..._snapshot_ensemble.ipynb`), los miembros salen de corridas completas distintas, así que exploran mínimos más
alejados entre sí.

**Preguntas, en orden:**

1. **Ensamble baseline** (H42 + H105): dos corridas de la *misma receta* (`cell_classifier_k70_haversine_aug.ipynb`)
   que solo difieren en la semilla. Mide la ganancia de "tener dos modelos", sin diversidad de receta. Comparar
   H105 contra H42 también estima la **variabilidad por semilla** de un modelo individual.
2. **Diversidad de receta**: ensambles haversine + focal y haversine + SAM. Si ganan más (sobre su mejor miembro)
   que el baseline, la receta distinta aporta diversidad útil más allá de cambiar la semilla.
3. **Referencia a superar**: el mejor modelo individual (S42, SAM), porque un ensamble que no le gana no sirve
   para jugar.

**Evaluación:** misma validación global que los demás notebooks, sin ajustar nada sobre ella (promedio simple, sin
pesos ni temperatura), así que la comparación entre ensambles y miembros es justa. Significancia con **bootstrap
pareado por panorama** (las 4 vistas de un panorama no son independientes): se remuestrean panoramas con
reposición y se recalculan las diferencias en accuracy top-1 y error mediano; todas las comparaciones usan los
mismos remuestreos.

Orden: **1. Configuración → 2. Logits de validación → 3. Métricas → 4. Diversidad → 5. Bootstrap → 6. Ganancia vs. costo**.

## 1. Configuración

In [ ]:
from pathlib import Path
from io import BytesIO
import json
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from datasets import Dataset as HFDataset, Image as HFImage
from PIL import Image
from tqdm.auto import tqdm
from IPython.display import display

# Miembros: nombre -> carpeta de la corrida (relativa a la raíz del proyecto o absoluta). Se usa su mejor checkpoint.
MIEMBROS = {
    "H42": "outputs/resnet18_IMAGENET_celdas_k70_haversine_aug/20261109_180324_467412",
    "H105": "outputs/resnet18_IMAGENET_celdas_k70_haversine_aug/20261112_134526_509022",
    "F42": "outputs/resnet18_IMAGENET_celdas_k70_haversine_aug_focal/20261111_122523_860807",
    "S42": "outputs/resnet18_IMAGENET_celdas_k70_haversine_aug_sam/20261112_114257_997479",
}
ARCHIVO_CHECKPOINT = "mejor_resnet18_celdas.pt"

# Ensambles a evaluar: nombre -> miembros (promedio simple de probabilidades).
ENSAMBLES = {
    "H42+H105": ["H42", "H105"],  # Baseline: misma receta, distinta semilla.
    "H42+F42": ["H42", "F42"],
    "H105+F42": ["H105", "F42"],
    "H42+S42": ["H42", "S42"],
    "H105+S42": ["H105", "S42"],
    "F42+S42": ["F42", "S42"],
    "H42+F42+S42": ["H42", "F42", "S42"],
    "todos (4)": ["H42", "H105", "F42", "S42"],
}
REFERENCIA = "S42"  # Mejor modelo individual: todo ensamble se compara también contra él.
# Comparaciones extra para el bootstrap (A − B), además de "ensamble − su mejor miembro" y "ensamble − REFERENCIA".
COMPARACIONES_EXTRA = [("H105", "H42"), ("H42+F42", "H42+H105"), ("H105+F42", "H42+H105"),
                       ("H42+S42", "H42+H105"), ("H105+S42", "H42+H105")]

NOMBRE_SALIDA = "ensamble_corridas_k70"  # Carpeta dentro de outputs/.
RUTA_VALIDACION = "datasets/samples/data/validation/validation.parquet"
COL_CLUSTER = "ind_cluster"
COLS_COORDENADAS = ["latitude", "longitude"]
RADIO_TIERRA_KM = 6371.0
N_BINS_ECE = 10
B_BOOTSTRAP = 2000
BATCH_SIZE = 64
NUM_WORKERS = 0
SEMILLA = 42
DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"

raiz = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), None)
if raiz is None:
    raise FileNotFoundError("Ejecutá el notebook dentro del proyecto.")


def resolver_ruta(ruta):
    p = Path(ruta).expanduser()
    return (p if p.is_absolute() else raiz / p).resolve()


rutas_miembros = {n: resolver_ruta(r) for n, r in MIEMBROS.items()}
for nombre, ruta in rutas_miembros.items():
    for archivo in (ruta / ARCHIVO_CHECKPOINT, ruta / "prototipos.csv", ruta / "predicciones_validation.csv"):
        if not archivo.is_file():
            raise FileNotFoundError(f"{nombre}: falta {archivo}")
for nombre, ms in ENSAMBLES.items():
    if len(ms) < 2 or len(set(ms)) != len(ms) or not set(ms) <= set(MIEMBROS):
        raise ValueError(f"Ensamble {nombre} inválido: {ms}")
nombres_validos = set(MIEMBROS) | set(ENSAMBLES)
for a, b in COMPARACIONES_EXTRA:
    if not {a, b} <= nombres_validos:
        raise ValueError(f"Comparación con nombres desconocidos: {a}, {b}")
assert REFERENCIA in MIEMBROS
ruta_validacion = resolver_ruta(RUTA_VALIDACION)
ruta_salida = raiz / "outputs" / NOMBRE_SALIDA
ruta_salida.mkdir(parents=True, exist_ok=True)
device = torch.device(DISPOSITIVO)
(ruta_salida / "parametros.json").write_text(json.dumps(
    {"miembros": MIEMBROS, "checkpoint": ARCHIVO_CHECKPOINT, "ensambles": ENSAMBLES, "referencia": REFERENCIA,
     "b_bootstrap": B_BOOTSTRAP, "semilla": SEMILLA}, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Salida: {ruta_salida}\nDispositivo: {device}")

### Compatibilidad entre miembros

Para que promediar tenga sentido, todos los miembros tienen que usar **las mismas celdas** (mismos prototipos,
mismo hash), la misma arquitectura y el mismo preprocesamiento de validación. Se verifica con la configuración
guardada en cada checkpoint.

In [ ]:
checkpoints, configs = {}, {}
for nombre, ruta in rutas_miembros.items():
    ckpt = torch.load(ruta / ARCHIVO_CHECKPOINT, map_location="cpu", weights_only=True)
    checkpoints[nombre], configs[nombre] = ckpt, ckpt["config"]

ref_cfg = configs[REFERENCIA]
for nombre, cfg in configs.items():
    for clave in ("arquitectura", "tarea", "num_clases", "dropout", "tamano_imagen", "preprocesamiento_val"):
        if cfg.get(clave) != ref_cfg.get(clave):
            raise ValueError(f"{nombre} difiere de {REFERENCIA} en '{clave}': {cfg.get(clave)} vs {ref_cfg.get(clave)}")
    if cfg["prototipos"]["sha256"] != ref_cfg["prototipos"]["sha256"]:
        raise ValueError(f"{nombre} usa otros prototipos.")
    if Path(cfg["ruta_validation"]).name != ruta_validacion.name:
        raise ValueError(f"{nombre} se validó con otro archivo: {cfg['ruta_validation']}")

resumen_miembros = pd.DataFrame({n: {
    "semilla": c.get("semilla"), "loss": c.get("loss"), "gamma_focal": c.get("gamma_focal", 0),
    "optimizador": c.get("optimizador", "AdamW"), "rho_sam": c.get("rho_sam", 0),
    "epoca_checkpoint": checkpoints[n]["epoca"]} for n, c in configs.items()}).T
display(resumen_miembros)

NUM_CLASES, TAMANO_IMAGEN, DROPOUT = ref_cfg["num_clases"], ref_cfg["tamano_imagen"], ref_cfg["dropout"]
prototipos = pd.read_csv(rutas_miembros[REFERENCIA] / "prototipos.csv").sort_values(COL_CLUSTER).reset_index(drop=True)
assert len(prototipos) == NUM_CLASES and (prototipos[COL_CLUSTER].to_numpy() == np.arange(NUM_CLASES)).all()
coords_prototipos = prototipos[["lat", "lon"]].to_numpy(dtype=np.float64)


def to_xyz(lat, lon):
    lat, lon = np.radians(lat), np.radians(lon)
    return np.column_stack([np.cos(lat) * np.cos(lon), np.cos(lat) * np.sin(lon), np.sin(lat)])


def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.deg2rad, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * RADIO_TIERRA_KM * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


dist_proto = haversine_km(coords_prototipos[:, None, 0], coords_prototipos[:, None, 1],
                          coords_prototipos[None, :, 0], coords_prototipos[None, :, 1])

cache_hf = str(raiz / ".cache/modeling/hf_celdas")
datos_val = HFDataset.from_parquet(str(ruta_validacion), cache_dir=cache_hf).cast_column("image", HFImage(decode=False))
meta_val = datos_val.select_columns(["image_id", "panoid", "country_code", *COLS_COORDENADAS]).to_pandas()
lat, lon = (meta_val[c].to_numpy(dtype=np.float64) for c in COLS_COORDENADAS)
real = (to_xyz(lat, lon) @ to_xyz(coords_prototipos[:, 0], coords_prototipos[:, 1]).T).argmax(axis=1)
dist_val = haversine_km(lat[:, None], lon[:, None], coords_prototipos[None, :, 0], coords_prototipos[None, :, 1])
assert (dist_val.argmin(axis=1) == real).all()
print(f"Validación: {len(meta_val):,} imágenes, {meta_val['panoid'].nunique():,} panoramas")

## 2. Logits de validación

Cada checkpoint se evalúa una sola vez y sus logits se guardan **en la carpeta de su corrida**
(`logits_validation_<checkpoint>.npy`), así otros ensambles los reutilizan sin pasar por la GPU.

**Control:** el argmax de los logits recalculados tiene que coincidir con `cluster_predicho` de las predicciones
que guardó cada corrida, y la accuracy con la reportada. Si no coincide, algo del preprocesamiento o del orden
de la validación cambió y el ensamble no sería válido.

In [ ]:
pesos_imagenet = models.ResNet18_Weights.IMAGENET1K_V1.transforms()
transform_val = transforms.Compose([
    transforms.Resize(TAMANO_IMAGEN), transforms.CenterCrop(TAMANO_IMAGEN), transforms.ToTensor(),
    transforms.Normalize(pesos_imagenet.mean, pesos_imagenet.std),
])
if repr(transform_val) != ref_cfg["preprocesamiento_val"]:
    raise ValueError("El preprocesamiento de validación no coincide con el de las corridas.")


class ImagenesVal(Dataset):
    def __init__(self, datos, transform):
        self.datos, self.transform = datos, transform

    def __len__(self):
        return len(self.datos)

    def __getitem__(self, idx):
        contenido = self.datos[int(idx)]["image"]
        fuente = BytesIO(contenido["bytes"]) if contenido.get("bytes") is not None else resolver_ruta(contenido["path"])
        with Image.open(fuente) as imagen:
            return self.transform(imagen.convert("RGB"))


loader_val = DataLoader(ImagenesVal(datos_val, transform_val), batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=device.type == "cuda")


def construir_modelo():
    m = models.resnet18(weights=None)
    m.fc = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(m.fc.in_features, NUM_CLASES))
    return m.to(device)


@torch.inference_mode()
def calcular_logits(state_dict, nombre):
    m = construir_modelo()
    m.load_state_dict(state_dict)
    m.eval()
    salida = torch.cat([m(x.to(device, non_blocking=True)).float().cpu() for x in tqdm(loader_val, desc=nombre)])
    del m
    return salida.numpy().astype(np.float64)


logits = {}
controles = {}
for nombre, ruta in rutas_miembros.items():
    ruta_logits = ruta / f"logits_validation_{Path(ARCHIVO_CHECKPOINT).stem}.npy"
    if not ruta_logits.is_file():
        np.save(ruta_logits, calcular_logits(checkpoints[nombre]["model_state_dict"], nombre))
    logits[nombre] = np.load(ruta_logits)
    if logits[nombre].shape != (len(meta_val), NUM_CLASES):
        raise ValueError(f"{nombre}: logits con forma {logits[nombre].shape}")
    guardadas = pd.read_csv(ruta / "predicciones_validation.csv").set_index("image_id").loc[meta_val["image_id"]]
    pred = logits[nombre].argmax(axis=1)
    coincidencia = (pred == guardadas["cluster_predicho"].to_numpy()).mean()
    acc_guardada = json.loads((ruta / "metricas_validation.json").read_text(encoding="utf-8"))["accuracy_top1"]
    controles[nombre] = {"coincidencia_argmax": coincidencia, "accuracy_recalculada": (pred == real).mean(),
                         "accuracy_reportada": acc_guardada}
    # Tolerancia mínima por no determinismo de cuDNN en empates casi exactos.
    if coincidencia < 0.995 or abs((pred == real).mean() - acc_guardada) > 2e-3:
        raise ValueError(f"{nombre}: los logits recalculados no reproducen la corrida ({controles[nombre]}).")
del checkpoints
display(pd.DataFrame(controles).T.round(4))

## 3. Métricas de miembros y ensambles

Para cada miembro y ensamble: accuracy top-1/top-5, NLL, ECE, error en km del argmax y de la predicción por mínimo
riesgo (MBR: la celda que minimiza la distancia esperada). Para los ensambles se agregan la ganancia en accuracy y
en error mediano **sobre su mejor miembro** (el de mayor accuracy) y **sobre el promedio de sus miembros**.

In [ ]:
def softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def ece_top_label(p, y, n_bins=N_BINS_ECE):
    conf, acierto = p.max(axis=1), p.argmax(axis=1) == y
    bin_id = np.clip(np.digitize(conf, np.linspace(0, 1, n_bins + 1)[1:-1]), 0, n_bins - 1)
    return float(sum((bin_id == b).mean() * abs(acierto[bin_id == b].mean() - conf[bin_id == b].mean())
                     for b in range(n_bins) if (bin_id == b).any()))


filas_idx = np.arange(len(real))


def predicciones(p):
    pred, pred_mbr = p.argmax(axis=1), (p @ dist_proto).argmin(axis=1)
    return {"pred": pred, "pred_mbr": pred_mbr, "acierto": pred == real,
            "km": dist_val[filas_idx, pred], "km_mbr": dist_val[filas_idx, pred_mbr]}


def metricas(p):
    r = predicciones(p)
    top5 = np.argsort(-p, axis=1)[:, :5]
    return {"accuracy_top1": r["acierto"].mean(), "accuracy_top5": (top5 == real[:, None]).any(axis=1).mean(),
            "nll": -np.log(np.clip(p[filas_idx, real], 1e-12, None)).mean(), "ece": ece_top_label(p, real),
            "error_medio_km": r["km"].mean(), "error_mediano_km": np.median(r["km"]),
            "accuracy_top1_mbr": (r["pred_mbr"] == real).mean(),
            "error_medio_km_mbr": r["km_mbr"].mean(), "error_mediano_km_mbr": np.median(r["km_mbr"])}


probs = {n: softmax(z) for n, z in logits.items()}
for nombre, ms in ENSAMBLES.items():
    probs[nombre] = np.mean([probs[m] for m in ms], axis=0)
tabla = pd.DataFrame({n: metricas(p) for n, p in probs.items()}).T
tabla.insert(0, "miembros", [1 if n in MIEMBROS else len(ENSAMBLES[n]) for n in tabla.index])

mejor_miembro = {n: max(ms, key=lambda m: tabla.at[m, "accuracy_top1"]) for n, ms in ENSAMBLES.items()}
for n, ms in ENSAMBLES.items():
    tabla.at[n, "mejor_miembro"] = mejor_miembro[n]
    tabla.at[n, "ganancia_acc_vs_mejor"] = tabla.at[n, "accuracy_top1"] - tabla.at[mejor_miembro[n], "accuracy_top1"]
    tabla.at[n, "ganancia_acc_vs_promedio"] = tabla.at[n, "accuracy_top1"] - tabla.loc[ms, "accuracy_top1"].mean()
    tabla.at[n, "ganancia_km_vs_mejor"] = tabla.at[n, "error_mediano_km"] - tabla.at[mejor_miembro[n], "error_mediano_km"]
tabla.to_csv(ruta_salida / "metricas_miembros_y_ensambles.csv")
columnas = ["miembros", "accuracy_top1", "accuracy_top5", "nll", "ece", "error_mediano_km", "error_medio_km",
            "accuracy_top1_mbr", "error_mediano_km_mbr", "ganancia_acc_vs_mejor", "ganancia_acc_vs_promedio",
            "ganancia_km_vs_mejor"]
display(tabla[columnas].round(4))

## 4. Diversidad entre miembros

Un ensamble solo mejora si sus miembros se equivocan en imágenes distintas. Se miden el **desacuerdo de argmax**
(fracción de imágenes en que predicen distinta celda), la **correlación de las salidas softmax** y la
**fracción de errores compartidos** (de las imágenes que falla alguno de los dos, en cuántas fallan ambos).

Atención: H42 y F42 comparten semilla (misma inicialización de la cabeza y mismo orden de batches) y S42 también
usa la semilla 42; H105 es la única con otra semilla. Comparar H42+F42 con H105+F42 separa el efecto de la receta
del de compartir semilla.

In [ ]:
nombres_m = list(MIEMBROS)
M = len(nombres_m)
pred_m = {n: probs[n].argmax(axis=1) for n in nombres_m}
falla = {n: pred_m[n] != real for n in nombres_m}
desacuerdo = pd.DataFrame([[(pred_m[a] != pred_m[b]).mean() for b in nombres_m] for a in nombres_m],
                          index=nombres_m, columns=nombres_m)
correlacion = pd.DataFrame(np.corrcoef(np.stack([probs[n].ravel() for n in nombres_m])),
                           index=nombres_m, columns=nombres_m)
errores_compartidos = pd.DataFrame([[(falla[a] & falla[b]).sum() / (falla[a] | falla[b]).sum() for b in nombres_m]
                                    for a in nombres_m], index=nombres_m, columns=nombres_m)
for titulo, t in (("Desacuerdo de argmax", desacuerdo), ("Correlación de softmax", correlacion),
                  ("Errores compartidos (ambos fallan / alguno falla)", errores_compartidos)):
    print(titulo)
    display(t.round(3))
desacuerdo.to_csv(ruta_salida / "desacuerdo_argmax.csv")
correlacion.to_csv(ruta_salida / "correlacion_softmax.csv")
errores_compartidos.to_csv(ruta_salida / "errores_compartidos.csv")

# Ganancia de cada ensamble de 2 contra el desacuerdo entre sus miembros.
pares = {n: ms for n, ms in ENSAMBLES.items() if len(ms) == 2}
fig, ax = plt.subplots(figsize=(6.5, 4.5))
for n, (a, b) in pares.items():
    x, y = desacuerdo.at[a, b], tabla.at[n, "ganancia_acc_vs_promedio"] * 100
    ax.scatter(x, y, color="tab:red" if n == "H42+H105" else "tab:blue")
    ax.annotate(n, (x, y), textcoords="offset points", xytext=(4, 4), fontsize=8)
ax.set(xlabel="desacuerdo de argmax entre los miembros", ylabel="ganancia de accuracy vs. promedio de miembros (pts)",
       title="Ensambles de 2: ganancia vs. diversidad (rojo = baseline)")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(ruta_salida / "ganancia_vs_desacuerdo.png", dpi=160)
plt.show()

## 5. Bootstrap pareado por panorama

Diferencias A − B en accuracy top-1 (puntos) y error mediano (km, argmax), con intervalo del 95%. Comparaciones:
cada ensamble contra **su mejor miembro** y contra la **referencia** (mejor modelo individual), más las de
`COMPARACIONES_EXTRA` (variabilidad por semilla y ensambles de receta distinta contra el baseline). Si el
intervalo no contiene el 0, la diferencia es significativa *respecto de la variabilidad de las imágenes*; no
incluye la variabilidad por semilla, que se ve aparte en H105 − H42.

In [ ]:
comparaciones = []
for n in ENSAMBLES:
    comparaciones.append((n, mejor_miembro[n]))
    if REFERENCIA not in ENSAMBLES[n]:
        comparaciones.append((n, REFERENCIA))
comparaciones += [c for c in COMPARACIONES_EXTRA if c not in comparaciones]

res = {n: predicciones(p) for n, p in probs.items()}
_, inverso = np.unique(meta_val["panoid"].to_numpy(), return_inverse=True)
n_pan = inverso.max() + 1
filas_por_panorama = [np.flatnonzero(inverso == i) for i in range(n_pan)]
rng = np.random.default_rng(SEMILLA)
difs = np.empty((B_BOOTSTRAP, len(comparaciones), 2))
for b in tqdm(range(B_BOOTSTRAP), desc="Bootstrap"):
    ix = np.concatenate([filas_por_panorama[i] for i in rng.integers(0, n_pan, n_pan)])
    acc = {n: r["acierto"][ix].mean() for n, r in res.items()}
    med = {n: np.median(r["km"][ix]) for n, r in res.items()}
    for j, (a, c) in enumerate(comparaciones):
        difs[b, j] = acc[a] - acc[c], med[a] - med[c]

filas = []
for j, (a, c) in enumerate(comparaciones):
    d_acc = res[a]["acierto"].mean() - res[c]["acierto"].mean()
    d_km = np.median(res[a]["km"]) - np.median(res[c]["km"])
    lo, hi = np.percentile(difs[:, j], [2.5, 97.5], axis=0)
    filas.append({"comparacion": f"{a} − {c}", "dif_accuracy_pts": 100 * d_acc,
                  "ic95_acc_inf": 100 * lo[0], "ic95_acc_sup": 100 * hi[0],
                  "acc_significativa": lo[0] > 0 or hi[0] < 0,
                  "dif_mediana_km": d_km, "ic95_km_inf": lo[1], "ic95_km_sup": hi[1],
                  "km_significativa": lo[1] > 0 or hi[1] < 0,
                  "ganadas": int((res[a]["acierto"] & ~res[c]["acierto"]).sum()),
                  "perdidas": int((~res[a]["acierto"] & res[c]["acierto"]).sum())})
bootstrap = pd.DataFrame(filas).set_index("comparacion")
bootstrap.to_csv(ruta_salida / "bootstrap_comparaciones.csv")
display(bootstrap.round(2))

## 6. Ganancia vs. costo de entrenamiento

El modelo final se entrenará con ~2M imágenes, donde solo entran 1–2 corridas, así que lo que importa es la
**mejora por hora de GPU**. El costo de cada miembro es la suma de `segundos` de su `historial.csv` (incluye las
épocas posteriores a la mejor, hasta el early stopping); el de un ensamble, la suma de sus miembros. El snapshot
ensemble de H42 se agrega como punto de referencia si están sus logits (costo = H42 + sus ciclos).

In [ ]:
horas = {n: pd.read_csv(r / "historial.csv")["segundos"].sum() / 3600 for n, r in rutas_miembros.items()}
costo = pd.DataFrame({"horas_gpu": {**horas, **{n: sum(horas[m] for m in ms) for n, ms in ENSAMBLES.items()}}})
costo = costo.join(tabla[["miembros", "accuracy_top1", "error_mediano_km"]])

# Referencia: snapshot ensemble de H42 (5 miembros, misma validación).
ruta_snap = rutas_miembros["H42"] / "snapshot_ensemble"
if (ruta_snap / "metricas_ensamble.json").is_file() and (ruta_snap / "logits_4.npy").is_file():
    p_snap = np.mean([softmax(np.load(ruta_snap / f)) for f in
                      ["logits_0_base.npy", "logits_1.npy", "logits_2.npy", "logits_3.npy", "logits_4.npy"]], axis=0)
    m_snap = metricas(p_snap)
    # Costo aproximado de los ciclos: tiempos de modificación de los snapshots (los ciclos corrieron seguidos).
    tiempos = sorted((ruta_snap / f"snapshot_{k}.pt").stat().st_mtime for k in range(1, 5))
    t0 = (ruta_snap / "logits_0_base.npy").stat().st_mtime
    costo.loc["snapshots H42 (5)"] = {"horas_gpu": horas["H42"] + (tiempos[-1] - t0) / 3600, "miembros": 5,
                                      "accuracy_top1": m_snap["accuracy_top1"],
                                      "error_mediano_km": m_snap["error_mediano_km"]}
costo.to_csv(ruta_salida / "ganancia_vs_costo.csv")
display(costo.round(3))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
for ax, col, titulo in ((axes[0], "accuracy_top1", "Accuracy top-1"), (axes[1], "error_mediano_km", "Error mediano (km)")):
    for n, fila in costo.iterrows():
        color = "tab:gray" if n in MIEMBROS else ("tab:green" if n.startswith("snapshots") else "tab:blue")
        ax.scatter(fila["horas_gpu"], fila[col], color=color)
        ax.annotate(n, (fila["horas_gpu"], fila[col]), textcoords="offset points", xytext=(4, 3), fontsize=8)
    ax.set(xlabel="horas de GPU de entrenamiento", title=f"{titulo} vs. costo (gris = individual, verde = snapshots)")
    ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(ruta_salida / "ganancia_vs_costo.png", dpi=160)
plt.show()
print(f"Resultados guardados en {ruta_salida}")